In [1]:
import sys
sys.path.append("../src")

import ast
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from scipy.stats import rankdata, spearmanr
from sentence_transformers import SentenceTransformer

from v2_eval import within_job_spearman

feat = pd.read_parquet("../data/processed/v2_features.parquet")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

BEST_THREE = ["semantic_similarity", "education_similarity", "skills_soft_coverage"]

feat["rank_within_job"] = feat.groupby("job_id")["matched_score"].transform(lambda s: rankdata(s) / len(s))

final_model = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", Ridge(alpha=1.0, random_state=42)),
])
final_model.fit(feat[BEST_THREE], feat["rank_within_job"])
print("Final v2 model trained on", len(feat), "rows.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Final v2 model trained on 9544 rows.


In [2]:
ext_df = pd.read_csv("hf://datasets/batuhanmtl/job_resume_fit/job_resume_fit.csv")
print(ext_df.shape)
ext_df.head(2)

(2385, 10)


,ID,resume_text,job_text,category,job_required_skills,resume_skill_list,ai_matched_skills,ai_match_score,skill_string_match_score,fuzzy_match_score
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,Human Resources Business Partner\n------------...,HR,"['human resources management', 'talent acquisi...","['customer service', 'team management', 'marke...","['human resources management', 'employee relat...",49.40,12.50,63.24
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...",Human Resources Business Partner\n------------...,HR,"['human resources management', 'talent acquisi...","['adobe photoshop', 'adp', 'asset management',...","['human resources management', 'talent acquisi...",77.64,16.67,59.07


In [3]:
def safe_list(x):
    try:
        items = ast.literal_eval(x)
        return [str(s).strip().lower() for s in items if str(s).strip()]
    except Exception:
        return []

resume_skills_ext = ext_df["resume_skill_list"].apply(safe_list)
job_skills_ext = ext_df["job_required_skills"].apply(safe_list)

In [4]:
def embed_series(texts, batch_size=64):
    clean = texts.fillna("").astype(str).str.strip()
    return embedder.encode(clean.tolist(), batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)

resume_vecs = embed_series(ext_df["resume_text"])
job_vecs = embed_series(ext_df["job_text"])
semantic_similarity_ext = np.einsum("ij,ij->i", resume_vecs, job_vecs)

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

In [5]:
all_skills_ext = sorted({s for l in resume_skills_ext for s in l} | {s for l in job_skills_ext for s in l})
print(len(all_skills_ext), "distinct skills (external)")

skill_vecs_ext = embedder.encode(all_skills_ext, batch_size=128, normalize_embeddings=True, show_progress_bar=True)
skill_idx_ext = {s: i for i, s in enumerate(all_skills_ext)}

def soft_coverage(c, r):
    if not c or not r:
        return np.nan
    C = skill_vecs_ext[[skill_idx_ext[s] for s in c]]
    R = skill_vecs_ext[[skill_idx_ext[s] for s in r]]
    return float((R @ C.T).max(axis=1).mean())

skills_soft_coverage_ext = [soft_coverage(c, r) for c, r in zip(resume_skills_ext, job_skills_ext)]

19467 distinct skills (external)


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

In [6]:
ext_feat = pd.DataFrame({
    "semantic_similarity": semantic_similarity_ext,
    "education_similarity": np.nan,
    "skills_soft_coverage": skills_soft_coverage_ext,
})

ext_df["predicted_v2"] = final_model.predict(ext_feat[BEST_THREE])
print(ext_df["predicted_v2"].describe())

count    2385.000000
mean        0.608314
std         0.087660
min         0.363364
25%         0.543510
50%         0.606257
75%         0.672469
max         0.843069
Name: predicted_v2, dtype: float64


In [7]:
comparisons = ["ai_match_score", "skill_string_match_score", "fuzzy_match_score"]

print("=== Overall correlations ===")
for col in comparisons:
    valid = ext_df[["predicted_v2", col]].dropna()
    r, _ = spearmanr(valid["predicted_v2"], valid[col])
    print(f"{col:30s} rho={r:.3f}")

print("\n=== Within-category correlations (comparable to the metric used in v2) ===")
for col in comparisons:
    r = within_job_spearman(ext_df[col], ext_df["predicted_v2"], ext_df["category"], min_rows=20)
    print(f"{col:30s} rho={r:.3f}")

print(f"\nv2 prediction standard deviation : {ext_df['predicted_v2'].std():.4f}  (v1 reference : 0.0176)")

=== Overall correlations ===
ai_match_score                 rho=0.706
skill_string_match_score       rho=0.569
fuzzy_match_score              rho=0.682

=== Within-category correlations (comparable to the metric used in v2) ===
ai_match_score                 rho=0.550
skill_string_match_score       rho=0.447
fuzzy_match_score              rho=0.532

v2 prediction standard deviation : 0.0877  (v1 reference : 0.0176)
